# 다층 퍼셉트론(MLP)을 활용한 XOR 문제 해결 시각화 튜토리얼

이 노트북은 XOR(배타적 논리합) 문제를 해결하기 위해 다층 퍼셉트론(Multi-Layer Perceptron, MLP)을 구성하고 학습하는 과정을 단계별로 시각화하여 보여준다.
XOR 문제는 선형 분리가 불가능한 대표적인 문제이며, 이를 해결하기 위해 은닉층(Hidden Layer)과 비선형 활성화 함수(Activation Function)가 필수적이다.


## 1. 입력 데이터 설명

모델이 학습할 입력 데이터 `X`와 정답 데이터 `Y`를 정의한다. 
입력이 다를 때만 `1`을 출력하고, 같을 때는 `0`을 출력하는 것이 XOR의 규칙이다.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx
import time

# ==========================================
# 1. 파일 업로드 및 데이터 로드 (Colab 환경)
# ==========================================

X = torch.FloatTensor([[0, 0], [0, 1], [1, 0], [1, 1]])
Y = torch.FloatTensor([[0],    [1],    [1],    [0]])

print("입력 데이터 X:\n", X)
print("정답 데이터 Y:\n", Y)



## 2. 하이퍼파라미터 설정 및 신경망 구조 시각화

학습률(Learning Rate), 총 에포크(Epoch) 수, 은닉층의 개수 및 각 은닉층의 노드 수, 그리고 활성화 함수를 설정한다.
설정된 값에 따라 신경망의 구조를 NN-SVG 형태로 동적으로 시각화하여 보여준다.
은닉층 레이어 수와 노드 수에 어떠한 큰 값이 들어와도 유연하게 대응하여 그래프를 그린다.


In [ ]:
# ==========================================
## 2. 하이퍼파라미터 설정
# ==========================================

learning_rate = 1.0
epochs = 10000
# 은닉층 노드 수 리스트 (예: [8] -> 1개의 은닉층, 8개의 노드)
hidden_layers = [8]
# 활성화 함수 설정 (예: nn.Sigmoid(), nn.ReLU() 등)
activation_func = nn.Sigmoid()

print(f"학습률: {learning_rate}")
print(f"목표 에포크: {epochs}")
print(f"은닉층 구조: {hidden_layers}")
print(f"활성화 함수: {activation_func.__class__.__name__}")

def draw_neural_net(input_size, hidden_layers, output_size):
    """
    설정된 파라미터에 따라 신경망 구조를 그린다.
    """
    layer_sizes = [input_size] + hidden_layers + [output_size]
    fig = plt.figure(figsize=(8, 6))
    ax = fig.gca()
    ax.axis('off')
    
    left, right, bottom, top = 0.1, 0.9, 0.1, 0.9
    v_spacing = (top - bottom) / float(max(layer_sizes))
    h_spacing = (right - left) / float(len(layer_sizes) - 1)
    
    # 노드 그리기
    for n, layer_size in enumerate(layer_sizes):
        layer_top = v_spacing * (layer_size - 1) / 2. + (top + bottom) / 2.
        for m in range(layer_size):
            circle = plt.Circle((n * h_spacing + left, layer_top - m * v_spacing), v_spacing / 4.,
                                color='w', ec='k', zorder=4)
            ax.add_artist(circle)
            
    # 간선 그리기
    for n, (layer_size_a, layer_size_b) in enumerate(zip(layer_sizes[:-1], layer_sizes[1:])):
        layer_top_a = v_spacing * (layer_size_a - 1) / 2. + (top + bottom) / 2.
        layer_top_b = v_spacing * (layer_size_b - 1) / 2. + (top + bottom) / 2.
        for m in range(layer_size_a):
            for o in range(layer_size_b):
                line = plt.Line2D([n * h_spacing + left, (n + 1) * h_spacing + left],
                                  [layer_top_a - m * v_spacing, layer_top_b - o * v_spacing], c='k')
                ax.add_artist(line)
    
    plt.title("Neural Network Architecture Visualization")
    plt.show()

draw_neural_net(2, hidden_layers, 1)



## 3. 첫 순전파(Forward Pass) 진행 및 시각화

설정한 하이퍼파라미터를 바탕으로 MLP 모델을 동적으로 생성한다.
초기화된 가중치를 사용하여 첫 번째 순전파를 진행한다.
입력 데이터가 각 층을 지날 때마다 데이터가 어떻게 변환되는지 각 레이어의 출력값을 시각적으로 확실하게 보여준다.


In [ ]:
# ==========================================
## 3. 다이내믹 모델 생성 및 첫 순전파
# ==========================================

class DynamicMLP(nn.Module):
    def __init__(self, input_size, hidden_layers, output_size, activation):
        super(DynamicMLP, self).__init__()
        self.layers = nn.ModuleList()
        self.activation = activation
        
        # 은닉층 생성
        in_size = input_size
        for h_size in hidden_layers:
            self.layers.append(nn.Linear(in_size, h_size))
            in_size = h_size
            
        # 출력층 생성
        self.output_layer = nn.Linear(in_size, output_size)
        self.output_activation = nn.Sigmoid() # 최종 출력은 0~1 사이
        
    def forward(self, x):
        activations = [] # 각 레이어의 출력값을 저장
        out = x
        for layer in self.layers:
            out = layer(out)
            out = self.activation(out)
            activations.append(out)
        out = self.output_layer(out)
        out = self.output_activation(out)
        activations.append(out)
        return out, activations

# 모델 인스턴스화
model = DynamicMLP(input_size=2, hidden_layers=hidden_layers, output_size=1, activation=activation_func)

# 첫 순전파 수행 (기울기 계산 필요 위해 detach 하지 않음)
predictions, activations = model(X)

print("[첫 순전파 결과]")
for i, act in enumerate(activations[:-1]):
    print(f"\n--- 은닉층 {i+1} 출력값 ---")
    print(act.detach().numpy())
    
print("\n--- 최종 출력층 출력값 ---")
print(predictions.detach().numpy())

# 시각화 (히트맵)
fig, axes = plt.subplots(1, len(activations), figsize=(4 * len(activations), 4))
if len(activations) == 1:
    axes = [axes]
for i, (act, ax) in enumerate(zip(activations, axes)):
    cax = ax.matshow(act.detach().numpy(), cmap='viridis', vmin=0, vmax=1)
    for (row, col), val in np.ndenumerate(act.detach().numpy()):
        ax.text(col, row, f"{val:.2f}", ha='center', va='center', color='white' if val < 0.5 else 'black')
    title = f"Hidden Layer {i+1} Output" if i < len(activations)-1 else "Final Output"
    ax.set_title(title)
    ax.set_xlabel("Nodes")
    ax.set_ylabel("Data Index")
plt.tight_layout()
plt.show()



## 4. 평균 제곱 오차(MSE) 계산 및 시각화

순전파를 통해 얻은 예측값(출력값)과 실제 정답(Y) 간의 오차를 계산한다.
이 과정에서 각 데이터 포인트의 오차(Error), 오차의 제곱(Squared Error), 그리고 최종 평균 오차(Mean Squared Error)로 나아가는 중간값들을 명확하게 보여준다.


In [ ]:
# ==========================================
## 4. 오차(Loss) 계산 과정 시각화
# ==========================================

# 손실 함수 정의 (Binary Cross Entropy 대신 직관적인 시각화를 위해 MSE 사용)
criterion = nn.MSELoss()

# 파이토치를 통한 최종 Loss 계산
loss = criterion(predictions, Y)

# 중간값 계산 (시각화용)
error = (predictions - Y).detach().numpy()
squared_error = error ** 2
mean_squared_error = squared_error.mean()

print("1. 각 데이터별 오차 (예측값 - 정답):")
print(error)

print("\n2. 각 데이터별 오차의 제곱 (오차^2):")
print(squared_error)

print(f"\n3. 최종 오차 (MSE, 오차 제곱의 평균): {mean_squared_error:.6f}")
print(f"   PyTorch의 Loss 값과 동일함: {loss.item():.6f}")

# 중간값 시각화 (막대 그래프)
fig, ax = plt.subplots(figsize=(8, 4))
x_axis = np.arange(len(Y))
width = 0.35

ax.bar(x_axis - width/2, error.flatten(), width, label='Error (Pred - Y)', color='coral')
ax.bar(x_axis + width/2, squared_error.flatten(), width, label='Squared Error', color='darkred')

ax.set_xticks(x_axis)
ax.set_xticklabels(['Data 1 (0,0)', 'Data 2 (0,1)', 'Data 3 (1,0)', 'Data 4 (1,1)'])
ax.axhline(0, color='black', linewidth=1)
ax.set_ylabel('Value')
ax.set_title(f'Error Distribution (Final MSE Loss: {loss.item():.6f})')
ax.legend()
plt.show()



## 5. 역전파(Backpropagation) 및 가중치 업데이트 시각화

계산된 오차(Loss)를 역방향으로 전파하여 각 가중치가 오차에 미치는 영향도(기울기, Gradient)를 적분(계산)한다.
이 기울기와 학습률(Learning Rate)을 곱하여 기존 가중치에 빼줌으로써 가중치를 업데이트한다.
업데이트 전 가중치, 계산된 기울기, 그리고 업데이트 후 가중치를 시각적으로 명확하게 비교하여 보여준다.


In [ ]:
# ==========================================
## 5. 역전파 및 가중치 업데이트
# ==========================================

optimizer = optim.SGD(model.parameters(), lr=learning_rate)

# 업데이트 전 가중치 저장 (첫 번째 은닉층 기준)
old_weights = model.layers[0].weight.data.clone().numpy()

# 역전파 수행
optimizer.zero_grad() # 기존 기울기 초기화
loss.backward()       # 역전파를 통한 기울기 계산

# 계산된 기울기 추출
gradients = model.layers[0].weight.grad.numpy()

# 가중치 업데이트 수행
optimizer.step()

# 업데이트 후 가중치 저장
new_weights = model.layers[0].weight.data.numpy()

print("[첫 번째 은닉층(Layer 1)의 가중치 변화]")

# 시각화 (히트맵 비교)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

def plot_heatmap(ax, data, title):
    cax = ax.matshow(data, cmap='coolwarm')
    for (row, col), val in np.ndenumerate(data):
        ax.text(col, row, f"{val:.3f}", ha='center', va='center', color='black')
    ax.set_title(title)
    ax.set_xlabel("Input Nodes")
    ax.set_ylabel("Hidden Nodes")

plot_heatmap(axes[0], old_weights, "1. Old Weights (Before Update)")
plot_heatmap(axes[1], gradients, "2. Gradients (Calculated by Backprop)")
plot_heatmap(axes[2], new_weights, "3. New Weights (After Update)")

plt.tight_layout()
plt.show()

print(f"가중치 업데이트 공식: New Weight = Old Weight - (Learning Rate * Gradient)")
print(f"예시 확인 (행 0, 열 0): {new_weights[0,0]:.3f} = {old_weights[0,0]:.3f} - ({learning_rate} * {gradients[0,0]:.3f})")



## 6. 업데이트 된 결과와 에포크 간 오차 비교

가중치가 업데이트된 상태에서 빠르게 두 번째 순전파를 수행한다.
이전 에포크(Epoch 1)의 오차와 다음 에포크(Epoch 2)의 오차가 얼마나 차이나는지 명확하게 볼 수 있게 시각화한다.


In [ ]:
# ==========================================
## 6. 두 번째 순전파 및 오차 비교
# ==========================================

# 두 번째 순전파 수행 (업데이트 된 가중치 사용)
new_predictions, _ = model(X)
new_loss = criterion(new_predictions, Y)

loss_epoch_1 = loss.item()
loss_epoch_2 = new_loss.item()

print(f"Epoch 1 오차: {loss_epoch_1:.6f}")
print(f"Epoch 2 오차: {loss_epoch_2:.6f}")
print(f"오차 변화량: {loss_epoch_1 - loss_epoch_2:.6f} 만큼 감소함.")

# 오차 비교 시각화 (막대 그래프)
fig, ax = plt.subplots(figsize=(6, 4))
epochs_labels = ['Epoch 1', 'Epoch 2']
losses = [loss_epoch_1, loss_epoch_2]

bars = ax.bar(epochs_labels, losses, color=['coral', 'lightblue'], width=0.5)

# 값 표기
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.005, f'{yval:.6f}', ha='center', va='bottom')

ax.set_ylabel('MSE Loss')
ax.set_title('Loss Comparison (Epoch 1 vs Epoch 2)')
ax.set_ylim(0, max(losses) * 1.2)
plt.show()



## 7. 본격적인 학습 및 딥러닝 결과 시각화

앞서 진행한 1~2 에포크의 과정은 학습의 원리를 이해하기 위한 시각화 단계였다.
이제 설정한 파라미터(`epochs` 등)를 그대로 사용하여 본격적으로 남은 에포크만큼 반복 학습을 진행한다.
학습이 끝난 후 Loss 감소 그래프, 분류 결과, 결정 경계(Decision Boundary) 등 딥러닝에서 볼 수 있는 시각화 자료들을 모두 보여준다.


In [ ]:
# ==========================================
## 7. 반복 학습 수행 및 결과 시각화 종합
# ==========================================

loss_history = [loss_epoch_1, loss_epoch_2]

# 진행률 출력 설정
print(f"총 {epochs} 에포크 학습 시작...\n")

# 이미 2에포크는 진행했으므로 나머지 에포크 진행
for epoch in range(2, epochs):
    optimizer.zero_grad()
    preds, _ = model(X)
    current_loss = criterion(preds, Y)
    current_loss.backward()
    optimizer.step()
    
    loss_history.append(current_loss.item())
    
    # 10% 단위로 출력
    if (epoch + 1) % (epochs // 10) == 0 or (epoch + 1) == epochs:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {current_loss.item():.6f}")

print("\n학습 완료!")

# 결과 시각화 종합 패널 구성
fig = plt.figure(figsize=(16, 5))

# 1. Loss Curve
ax1 = fig.add_subplot(1, 3, 1)
ax1.plot(loss_history, color='blue', linewidth=2)
ax1.set_title('Training Loss Curve')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('MSE Loss')
ax1.grid(True, linestyle='--', alpha=0.6)

# 2. 결정 경계(Decision Boundary) 시각화
ax2 = fig.add_subplot(1, 3, 2)
x_min, x_max = -0.2, 1.2
y_min, y_max = -0.2, 1.2
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                     np.linspace(y_min, y_max, 100))

grid_tensor = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])
with torch.no_grad():
    Z, _ = model(grid_tensor)
    Z = Z.numpy()
Z = Z.reshape(xx.shape)

contour = ax2.contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.8)
fig.colorbar(contour, ax=ax2)
scatter = ax2.scatter(X[:, 0], X[:, 1], c=Y[:, 0], cmap='RdBu', edgecolors='k', s=100)

for i in range(len(X)):
    ax2.text(X[i, 0], X[i, 1] + 0.05, f"({int(X[i,0].item())},{int(X[i,1].item())})\nTarget:{int(Y[i,0].item())}", 
             ha='center', va='bottom', fontsize=9, fontweight='bold')

ax2.set_title("Decision Boundary")
ax2.set_xlabel("Input X1")
ax2.set_ylabel("Input X2")

# 3. 최종 예측값 막대 그래프 비교
ax3 = fig.add_subplot(1, 3, 3)
final_preds, _ = model(X)
final_preds = final_preds.detach().numpy().flatten()
targets = Y.numpy().flatten()

x_axis = np.arange(len(targets))
width = 0.35

ax3.bar(x_axis - width/2, targets, width, label='Target', color='lightblue')
ax3.bar(x_axis + width/2, final_preds, width, label='Prediction', color='salmon')

ax3.set_xticks(x_axis)
ax3.set_xticklabels(['[0,0]', '[0,1]', '[1,0]', '[1,1]'])
ax3.set_title('Final Predictions vs Targets')
ax3.set_ylabel('Output Value')
ax3.legend()

plt.tight_layout()
plt.show()
